In [35]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# 1. Load the dataset
diabetes = pd.read_csv('diabetes.csv')

# 2. Separate features (X) and labels (y)
X = diabetes.drop('Outcome', axis=1)
y = diabetes['Outcome']

# 3. Split into training and testing sets FIRST to prevent data leakage
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42, stratify=y)

# 4. Create a Pipeline (Combines Scaling and the Model)
# This ensures that standard scaling is done perfectly inside cross-validation
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('svc', SVC(kernel='linear'))
])

# 5. Define Hyperparameters to tune
# The 'C' parameter controls regularization:
# - Low C: Strong regularization (prevents overfitting, risks underfitting)
# - High C: Weak regularization (prevents underfitting, risks overfitting)
param_grid = {
    'svc__C': [0.01, 0.1, 1, 10, 100]
}

# 6. Use GridSearchCV to find the best balance
# cv=5 means 5-fold cross-validation, ensuring the model generalizes well
grid_search = GridSearchCV(pipeline, param_grid, cv=5, scoring='accuracy')
grid_search.fit(X_train, y_train)

# Extract the best model from the grid search
best_model = grid_search.best_estimator_

print(f"Best Regularization Parameter (C): {grid_search.best_params_['svc__C']}")

# 7. Evaluate the best model
train_y_pred = best_model.predict(X_train)
test_y_pred = best_model.predict(X_test)

print("\n--- Model Evaluation ---")
print("Train set accuracy :", accuracy_score(y_train, train_y_pred))
print("Test set accuracy  :", accuracy_score(y_test, test_y_pred))
print("\nClassification Report (Test Data):\n", classification_report(y_test, test_y_pred))

# 8. Make a prediction for new input data
# Note: By using the pipeline, the input data is automatically scaled!
input_data = (1, 85, 66, 29, 0, 26.6, 0.351, 31)

# Convert to numpy array and reshape
np_arry_data = np.asarray(input_data).reshape(1, -1)

# Predict using the best_model pipeline
prediction = best_model.predict(np_arry_data)

print("\n--- New Prediction ---")
if prediction[0] == 1:
    print("This person has diabetes.")
else:
    print("This person does not have diabetes.")

Best Regularization Parameter (C): 0.1

--- Model Evaluation ---
Train set accuracy : 0.7879377431906615
Test set accuracy  : 0.7362204724409449

Classification Report (Test Data):
               precision    recall  f1-score   support

           0       0.76      0.86      0.81       165
           1       0.66      0.51      0.57        89

    accuracy                           0.74       254
   macro avg       0.71      0.68      0.69       254
weighted avg       0.73      0.74      0.73       254


--- New Prediction ---
This person does not have diabetes.


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
